### Overview


### Setup and Data Download

The following blocks of code will install the required packages and download the datasets to your Colab environment.

In [ ]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip install odc-stac rioxarray dask['distributed'] xvec exactextract

In [ ]:
import dask
import os
import duckdb
import geopandas as gpd
import matplotlib.pyplot as plt
from odc.stac import stac_load
from odc.geo.geobox import GeoBox
from odc.geo.xr import xr_reproject
import rioxarray as rxr
import xarray as xr
import xvec
import exactextract


In [ ]:
data_folder = 'data'
output_folder = 'output'

if not os.path.exists(data_folder):
    os.mkdir(data_folder)
if not os.path.exists(output_folder):
    os.mkdir(output_folder)

In [ ]:
def download(url):
    filename = os.path.join(data_folder, os.path.basename(url))
    if not os.path.exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)

census_folder = 'https://www2.census.gov/geo/tiger/GENZ2021/shp/'
zones_file = 'cb_2021_us_county_500k.zip'

for file in files:
  download(census_folder + file)

Setup a local Dask cluster. This distributes the computation across multiple workers on your computer.

In [ ]:
from dask.distributed import Client
client = Client()  # set up local cluster on the machine
client

If you are running this notebook in Colab, you will need to create and use a proxy URL to see the dashboard running on the local server.

In [ ]:
if 'google.colab' in str(get_ipython()):
    from google.colab import output
    port_to_expose = 8787  # This is the default port for Dask dashboard
    print(output.eval_js(f'google.colab.kernel.proxyPort({port_to_expose})'))


## Load Polygons

First we will read the  Zipped counties shapefile and filter out the counties that are in `California` state.

The dataframe has a column  as `STATE_NAME` having naesm of states that can be used to filter the counties for *California*.

In [ ]:
zones_file_path = os.path.join(data_folder, zones_file)

zones_df = gpd.read_file(zones_file_path)
california_gdf  = zones_df[zones_df['STATE_NAME'] == 'California'].copy()

california_gdf.head()

### Load Raster Data

We now load the GeoTIFF file using `rioxarray`.

In [ ]:
raster_file_path = 'https://storage.googleapis.com/spatialthoughts-public-data/ghsl/GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0_cog.tif'
da = rxr.open_rasterio(raster_file_path, chunks='auto', mask_and_scale=True)
da

In [ ]:
pop_da = da.squeeze()
pop_da

### Calculate Zonal Statistics

Reproject the polygons to match the projection of the population raster.

In [ ]:
california_gdf_reprojected = california_gdf.to_crs(pop_da.rio.crs)

Clip the raster to the bounds of the zones. `clip_box` is window-read aware with COGs — it only fetches the tiles that overlap the bounding box, so the data stays lazy until you call .compute().

In [ ]:
bounds = california_gdf_reprojected.total_bounds  # (minx, miny, maxx, maxy)
pop_da_clipped = pop_da.rio.clip_box(*bounds)
pop_da_clipped

In [ ]:
%%time
aggregated = pop_da_clipped.xvec.zonal_stats(
    california_gdf_reprojected.geometry, x_coords='x', y_coords='y', stats=['sum'],
)

In [ ]:
aggregated

In [ ]:
# Define attributes to be added
columns_dict = {"name": "NAME"}


def update_cube_attrs(ds, gdf, attrs_dict):
    for k, v in attrs_dict.items():
        ds[k] = (("geometry"), gdf[v].values)
        ds = ds.assign_coords({k: ds[k]})
    return ds


vector_data_cube = update_cube_attrs(aggregated, california_gdf_reprojected, columns_dict)

In [ ]:
aggregated_gdf = aggregated.xvec.to_geodataframe(name='population_sum', geometry="geometry")
aggregated_gdf

 Reset the index to convert the multi-index into columns, and select and rename columns to prepare the output.


In [ ]:
output_gdf = aggregated_gdf.reset_index()
output_gdf = output_gdf.rename(columns={'NAME': 'name', 'population_sum': 'population'})
output_gdf = output_gdf[['name', 'population', 'geometry']]
output_gdf

In [ ]:
# Define the output path
output_path = os.path.join(output_folder, 'california_counties_population.gpkg')

# Save the GeoDataFrame to a GeoPackage file
output_gdf.to_file(output_path, driver='GPKG')


## Exercise

[CHIRPS](https://www.chc.ucsb.edu/data/chirps3) - Climate Hazards Center InfraRed Precipitation with Station data - is gridded rainfall time-series data with coverage between 60°N to 60°S latitudes. The data is available aggregated over various time-periods.

The annual GeoTIFF files are available at https://data.chc.ucsb.edu/products/CHIRPS/v3.0/annual/global/tifs/.

Pick a year and calculate the average rainfall in each county in california.